In [ ]:
%%capture
!pip install llama_index pyvis Ipython langchain pypdf
!pip install llama-index-llms-huggingface
!pip install llama-index-embeddings-langchain

In [ ]:
!llama-index==0.10.33
!llama-index-core==0.10.33
!llama-index-embeddings-langchain==0.1.2
!llama-index-embeddings-openai==0.1.9
!llama-index-legacy==0.9.48
!llama-index-llms-huggingface==0.1.4
!langchain==0.1.16
!langchain-community==0.0.34
!langchain-core==0.1.46

In [ ]:
!pip install -U langchain-community

In [ ]:
import logging
import sys
#
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import KnowledgeGraphIndex
from llama_index.core import Settings
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import StorageContext
# from llama_index.llms.huggingface import HuggingFaceInferenceAPI
from llama_index.llms.huggingface import HuggingFaceLLM
from langchain.embeddings import HuggingFaceEmbeddings
# Import HuggingFaceInferenceAPIEmbeddings
from langchain.embeddings.huggingface import HuggingFaceInferenceAPIEmbeddings
from llama_index.embeddings.langchain import LangchainEmbedding
from pyvis.network import Network

In [ ]:
!pip install --upgrade llama-index
!pip install --upgrade llama-index-llms-huggingface
!pip install --upgrade llama-index-embeddings-huggingface
!pip install --upgrade transformers huggingface_hub


In [ ]:
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import ServiceContext, SimpleDirectoryReader, VectorStoreIndex

model_name = "HuggingFaceH4/zephyr-7b-beta"

llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.7}
)

embed_model = HuggingFaceEmbedding(
    model_name="thenlper/gte-large"
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install docx2txt

In [ ]:
import os
os.makedirs("/content/Documents", exist_ok=True)

!mv /content/medicine_data_info.docx /content/Documents/

In [ ]:
documents = SimpleDirectoryReader("/content/Documents").load_data()
print(len(documents))

In [ ]:
Settings.llm = llm
Settings.chunk_size = 512

graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

index = KnowledgeGraphIndex.from_documents( documents=documents,
                                           max_triplets_per_chunk=3,
                                           storage_context=storage_context,
                                           embed_model=embed_model,
                                          include_embeddings=True)

In [ ]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 22.3 MB/s eta 0:00:00


In [ ]:
from neo4j import GraphDatabase

uri = "neo4j+s://.databases.neo4j.io"
username = "neo4j"
password = ""

driver = GraphDatabase.driver(uri, auth=(username, password))


In [ ]:
def add_triplet(tx, subject, predicate, obj):
    query = (
        "MERGE (s:Entity {name: $subject}) "
        "MERGE (o:Entity {name: $object}) "
        "MERGE (s)-[r:" + predicate.upper().replace(" ", "_") + "]->(o)"
    )
    tx.run(query, subject=subject, object=obj)

with driver.session() as session:
    graph = index.get_networkx_graph()
    for source, target, data in graph.edges(data=True):
        predicate = data.get("predicate", "RELATED_TO")
        session.write_transaction(add_triplet, source, predicate, target)


In [ ]:
query = "What is paracetamol"
query_engine = index.as_query_engine(include_text=True,
                                     response_mode ="tree_summarize",
                                     embedding_mode="hybrid",
                                     similarity_top_k=5,)

message_template =f"""<|system|>Please check if the following pieces of context has any mention of the  keywords provided in the Question.If not then don't know the answer, just say that you don't know.Stop there.Please donot try to make up an answer.</s>
<|user|>
Question: {query}
Helpful Answer:
</s>"""

response = query_engine.query(message_template)

print(response.response.split("<|assistant|>")[-1].strip())